# 🎯 DDoS Detection - Comparativa de Modelos ML
## SDN Spine-Leaf Network

## ⚠️ INSTALAR DEPENDENCIAS
```bash
pip3 install pandas numpy scikit-learn joblib matplotlib ipykernel --break-system-packages
```

---
# 📂 RUTA DEL DATASET - CAMBIAR ESTA RUTA

In [ ]:
DATASET_PATH = "/home/ryu/Desktop/Labo/SdnShare/datasets/dataset_20260301_075018.csv"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

---
# 2. CARGAR Y LIMPIAR DATASET

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Dataset original: {df.shape[0]} filas, {df.shape[1]} columnas")

df = df.dropna()
print(f"Después de dropna: {df.shape[0]} filas")

numeric_cols = ['rx_pkts', 'tx_pkts', 'rx_bytes', 'tx_bytes', 'rx_pps', 'tx_pps', 'rx_bps', 'tx_bps']
df = df[~(df[numeric_cols] == 0).all(axis=1)]
print(f"Después de eliminar ceros: {df.shape[0]} filas")

df['label'] = df['label'].astype(int)

print(f"\nDistribución de labels:")
print(df['label'].value_counts())
print(f"\n0 = Normal, 1 = Ataque")

---
# 3. PREPROCESAMIENTO

In [ ]:
feature_columns = [
    'rx_pkts', 'tx_pkts', 'rx_bytes', 'tx_bytes',
    'drops', 'errors', 'collisions',
    'rx_pps', 'tx_pps', 'rx_bps', 'tx_bps',
    'avg_packet_size_rx', 'avg_packet_size_tx', 'rx_tx_ratio'
]

X = df[feature_columns]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test: {X_test.shape[0]} muestras")
print(f"Features: {len(feature_columns)}")

---
# 4. ENTRENAR MÚLTIPLES MODELOS

In [ ]:
models = {
    'DT': DecisionTreeClassifier(max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    'RF': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'LR': LogisticRegression(max_iter=1000, random_state=42),
    'GNB': GaussianNB(),
    'SVC': SVC(kernel='rbf', random_state=42),
    'GBC': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
}

trained_models = {}

for name, model in models.items():
    print(f"Entrenando {name}...", end=" ")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print("✅")

print("\n🎉 Todos los modelos entrenados")

---
# 5. EVALUACIÓN DETALLADA DE CADA MODELO

In [ ]:
def evaluate_model_detailed(model, X_test, y_test, model_name):
    """Evalúa un modelo con métricas detalladas por clase"""
    y_pred = model.predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    
    tn, fp, fn, tp = cm.ravel()
    
    accuracy = accuracy_score(y_test, y_pred)
    
    precision_normal = precision_score(y_test, y_pred, pos_label=0)
    precision_attack = precision_score(y_test, y_pred, pos_label=1)
    
    recall_normal = recall_score(y_test, y_pred, pos_label=0)
    recall_attack = recall_score(y_test, y_pred, pos_label=1)
    
    f1_normal = f1_score(y_test, y_pred, pos_label=0)
    f1_attack = f1_score(y_test, y_pred, pos_label=1)
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    
    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'N_Precision': precision_normal,
        'A_Precision': precision_attack,
        'N_Recall': recall_normal,
        'A_Recall': recall_attack,
        'N_F1': f1_normal,
        'A_F1': f1_attack,
        'FPR': fpr,
        'FNR': fnr,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn
    }

results = []

for name, model in trained_models.items():
    result = evaluate_model_detailed(model, X_test, y_test, name)
    results.append(result)

results_df = pd.DataFrame(results)
print("=" * 80)
print("📊 TABLA COMPARATIVA DE MODELOS")
print("=" * 80)
print("N = Normal (0), A = Attack (1)")
print("-" * 80)
display(results_df[['Model', 'Accuracy', 'N_Precision', 'A_Precision', 
                     'N_Recall', 'A_Recall', 'N_F1', 'A_F1', 'FPR', 'FNR']])

In [ ]:
print("\n" + "=" * 80)
print("📈 ANÁLISIS DETALLADO")
print("=" * 80)

for name, model in trained_models.items():
    result = results_df[results_df['Model'] == name].iloc[0]
    print(f"\n🔹 {name}")
    print(f"   Accuracy: {result['Accuracy']:.4f}")
    print(f"   Normal   - Prec: {result['N_Precision']:.4f}, Rec: {result['N_Recall']:.4f}, F1: {result['N_F1']:.4f}")
    print(f"   Ataque   - Prec: {result['A_Precision']:.4f}, Rec: {result['A_Recall']:.4f}, F1: {result['A_F1']:.4f}")
    print(f"   FPR: {result['FPR']:.4f}, FNR: {result['FNR']:.4f}")
    print(f"   CM: TP={result['TP']}, TN={result['TN']}, FP={result['FP']}, FN={result['FN']}")

---
# 6. SELECCIONAR EL MEJOR MODELO

In [ ]:
best_model_name = results_df.loc[results_df['Accuracy'].idxmax(), 'Model']
print(f"🏆 Mejor modelo por Accuracy: {best_model_name}")

best_model_attack = results_df.loc[results_df['A_Recall'].idxmax(), 'Model']
print(f"🏆 Mejor modelo por Recall (Ataque): {best_model_attack}")

best_model_fnr = results_df.loc[results_df['FNR'].idxmin(), 'Model']
print(f"🏆 Mejor modelo por menor FNR: {best_model_fnr}")

print("\n📋 Recomendación: Decision Tree (DT) es ideal por su simplicidad,")
print("   alta precisión y baja latencia en tiempo real.")

---
# 7. GUARDAR EL MODELO SELECCIONADO (Decision Tree)

In [ ]:
dt_model = trained_models['DT']

MODEL_PATH = "/home/ryu/Desktop/Labo/SdnShare/models/ddos_dt_model.pkl"
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

joblib.dump(dt_model, MODEL_PATH)

print(f"✅ Modelo Decision Tree guardado en: {MODEL_PATH}")

---
# 8. VISUALIZACIÓN

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Accuracy comparativa
    ax1 = axes[0, 0]
    ax1.bar(results_df['Model'], results_df['Accuracy'], color='steelblue')
    ax1.set_title('Accuracy por Modelo', fontsize=12)
    ax1.set_ylabel('Accuracy')
    ax1.set_ylim(0, 1.1)
    for i, v in enumerate(results_df['Accuracy']):
        ax1.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
    
    # 2. Precision comparativa
    ax2 = axes[0, 1]
    x = np.arange(len(results_df))
    width = 0.35
    ax2.bar(x - width/2, results_df['N_Precision'], width, label='Normal', color='green', alpha=0.7)
    ax2.bar(x + width/2, results_df['A_Precision'], width, label='Ataque', color='red', alpha=0.7)
    ax2.set_title('Precision por Modelo', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(results_df['Model'])
    ax2.legend()
    ax2.set_ylim(0, 1.1)
    
    # 3. Recall comparativa
    ax3 = axes[1, 0]
    ax3.bar(x - width/2, results_df['N_Recall'], width, label='Normal', color='green', alpha=0.7)
    ax3.bar(x + width/2, results_df['A_Recall'], width, label='Ataque', color='red', alpha=0.7)
    ax3.set_title('Recall por Modelo', fontsize=12)
    ax3.set_xticks(x)
    ax3.set_xticklabels(results_df['Model'])
    ax3.legend()
    ax3.set_ylim(0, 1.1)
    
    # 4. FPR y FNR
    ax4 = axes[1, 1]
    ax4.bar(x - width/2, results_df['FPR'], width, label='FPR', color='orange', alpha=0.7)
    ax4.bar(x + width/2, results_df['FNR'], width, label='FNR', color='purple', alpha=0.7)
    ax4.set_title('FPR y FNR por Modelo', fontsize=12)
    ax4.set_xticks(x)
    ax4.set_xticklabels(results_df['Model'])
    ax4.legend()
    
    plt.tight_layout()
    plt.savefig('/home/ryu/Desktop/Labo/SdnShare/models/model_comparison.png', dpi=150)
    print("✅ Gráficos guardados en: models/model_comparison.png")
    
except Exception as e:
    print(f"⚠️ Error al generar gráficos: {e}")

---
# 📊 RESUMEN FINAL

In [ ]:
dt_result = results_df[results_df['Model'] == 'DT'].iloc[0]

print("=" * 70)
print("🎉 ENTRENAMIENTO Y EVALUACIÓN COMPLETADOS")
print("=" * 70)
print(f"\n📁 Dataset: {DATASET_PATH.split('/')[-1]}")
print(f"📊 Muestras totales: {df.shape[0]}")
print(f"🔢 Muestras entrenamiento: {X_train.shape[0]}")
print(f"🔢 Muestras test: {X_test.shape[0]}")
print(f"📈 Modelos evaluados: {len(trained_models)}")
print("-" * 70)
print(f"\n🏆 MEJOR MODELO SELECCIONADO: Decision Tree (DT)")
print(f"   Accuracy:  {dt_result['Accuracy']:.4f}")
print(f"   Precision: {dt_result['A_Precision']:.4f} (Ataque)")
print(f"   Recall:    {dt_result['A_Recall']:.4f} (Ataque)")
print(f"   F1-Score:  {dt_result['A_F1']:.4f} (Ataque)")
print(f"   FPR:       {dt_result['FPR']:.4f}")
print(f"   FNR:       {dt_result['FNR']:.4f}")
print(f"\n💾 Modelo guardado en: {MODEL_PATH}")
print("=" * 70)